# logs_local_berk Leaderboard

Compare all runs from `logs_local_berk` (AI-GUIDED-CLEAN patch-based trainings on ROSIE TCGA): EarlyFusion, FiLM Rosie, FiLM ImageNet, etc.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis")
LOGS_LOCAL_BERK = PROJECT_ROOT / "AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk"

sys.path.insert(0, str(PROJECT_ROOT / "Datasets/pannuke_hf_cellvit/notebooks"))
from notebook_utils import load_runs_dataframe

In [2]:
runs_df = load_runs_dataframe(LOGS_LOCAL_BERK)
print(f"Loaded {len(runs_df)} runs")

Loaded 13 runs


## A) Overall Leaderboard (top 15 by mPQ)

In [3]:
lb = runs_df.sort_values("mPQ", ascending=False).head(15)
cols = ["run_name", "backbone", "method", "condition_source", "film_target", "mPQ", "bPQ", "DQ", "SQ", "Dice", "Jaccard", "F1"]
lb[[c for c in cols if c in lb.columns]]

,run_name,backbone,method,condition_source,film_target,mPQ,bPQ,DQ,SQ
1,2026-02-14T105042_FilmRosieWeights-samhbaseline,SAM-H,Baseline,Rosie,None,0.586468,0.667485,0.710377,0.729213
7,2026-02-14T154204_FilmIMAGENETWeights-samhbase...,SAM-H,Baseline,ImageNet,None,0.579894,0.665227,0.701391,0.730331
4,2026-02-14T105042_FilmRosieWeights-z4,SAM-H,FiLM,Rosie,z4,0.576011,0.661532,0.699588,0.719609
11,2026-02-17T103313_EarlyFusion-vec9-lr3e-05,SAM-H,EarlyFusion,None,None,0.575347,0.660660,0.691942,0.720563
8,2026-02-14T154204_FilmIMAGENETWeights-z4,SAM-H,FiLM,ImageNet,z4,0.569389,0.655707,0.686630,0.716776
3,2026-02-14T105042_FilmRosieWeights-z3z4,SAM-H,FiLM,Rosie,"z3,z4",0.567639,0.658371,0.682065,0.714946
0,2026-02-10T231234_FilmRosieWeights,SAM-H,FiLM,Rosie,None,0.566806,0.653192,0.683589,0.716408
6,2026-02-14T154156_FilmIMAGENETWeights-z1z4,SAM-H,FiLM,ImageNet,"z1,z2,z3,z4",0.558080,0.656999,0.675919,0.702241
10,2026-02-17T103313_EarlyFusion-vec9-lr1e-4,SAM-H,EarlyFusion,None,None,0.557498,0.659197,0.679708,0.705498
2,2026-02-14T105042_FilmRosieWeights-z1z4,SAM-H,FiLM,Rosie,"z1,z2,z3,z4",0.556498,0.658794,0.671127,0.687156


## B) Best per Group

In [4]:
grp = ["backbone", "method", "condition_source", "film_target"]
grp = [c for c in grp if c in runs_df.columns]
best_per = runs_df.loc[runs_df.groupby(grp)["mPQ"].idxmax()]
best_per = best_per.sort_values("mPQ", ascending=False)
best_per[["run_name"] + grp + ["mPQ"]].head(20)

,run_name,backbone,method,condition_source,film_target,mPQ
1,2026-02-14T105042_FilmRosieWeights-samhbaseline,SAM-H,Baseline,Rosie,None,0.586468
7,2026-02-14T154204_FilmIMAGENETWeights-samhbase...,SAM-H,Baseline,ImageNet,None,0.579894
4,2026-02-14T105042_FilmRosieWeights-z4,SAM-H,FiLM,Rosie,z4,0.576011
11,2026-02-17T103313_EarlyFusion-vec9-lr3e-05,SAM-H,EarlyFusion,None,None,0.575347
8,2026-02-14T154204_FilmIMAGENETWeights-z4,SAM-H,FiLM,ImageNet,z4,0.569389
3,2026-02-14T105042_FilmRosieWeights-z3z4,SAM-H,FiLM,Rosie,"z3,z4",0.567639
0,2026-02-10T231234_FilmRosieWeights,SAM-H,FiLM,Rosie,None,0.566806
6,2026-02-14T154156_FilmIMAGENETWeights-z1z4,SAM-H,FiLM,ImageNet,"z1,z2,z3,z4",0.558080
2,2026-02-14T105042_FilmRosieWeights-z1z4,SAM-H,FiLM,Rosie,"z1,z2,z3,z4",0.556498
5,2026-02-14T154138_FilmIMAGENETWeights-z3z4,SAM-H,FiLM,ImageNet,"z3,z4",0.552187


## C) Method Comparison: EarlyFusion vs FiLM vs Baseline

In [5]:
for method in runs_df["method"].dropna().unique():
    sub = runs_df[runs_df["method"] == method]
    best = sub.loc[sub["mPQ"].idxmax()]
    print(f"{method}: best mPQ={best['mPQ']:.4f} ({best['run_name']})")

FiLM: best mPQ=0.5760 (2026-02-14T105042_FilmRosieWeights-z4)
Baseline: best mPQ=0.5865 (2026-02-14T105042_FilmRosieWeights-samhbaseline)
EarlyFusion: best mPQ=0.5753 (2026-02-17T103313_EarlyFusion-vec9-lr3e-05)


## D) Apples-to-Apples: FiLM vs Baseline (by backbone)

In [6]:
baselines = runs_df[runs_df["method"] == "Baseline"].copy()
films = runs_df[runs_df["method"].isin(["FiLM", "ProxyFiLM", "LoRA+FiLM"])].copy()

rows = []
for bb in runs_df["backbone"].dropna().unique():
    base = baselines[baselines["backbone"] == bb]
    film = films[films["backbone"] == bb]
    if base.empty or film.empty:
        continue
    best_base = base.loc[base["mPQ"].idxmax()]
    best_film = film.loc[film["mPQ"].idxmax()]
    delta = best_film["mPQ"] - best_base["mPQ"]
    rows.append({
        "backbone": bb,
        "baseline_run": best_base["run_name"],
        "baseline_mPQ": best_base["mPQ"],
        "best_film_run": best_film["run_name"],
        "film_mPQ": best_film["mPQ"],
        "delta_mPQ": delta,
    })

if rows:
    delta_df = pd.DataFrame(rows).sort_values("delta_mPQ", ascending=False)
    delta_df
else:
    print("No backbone groups with both Baseline and FiLM runs.")

## E) Narrative Summary

In [7]:
best_run = runs_df.loc[runs_df["mPQ"].idxmax()]
print("Best overall run:", best_run["run_name"])
print("  mPQ =", round(best_run["mPQ"], 4))
print()

if rows:
    for _, r in delta_df.iterrows():
        print(f"Backbone {r['backbone']}:")
        print(f"  Baseline mPQ = {r['baseline_mPQ']:.4f}")
        print(f"  Best FiLM mPQ = {r['film_mPQ']:.4f}")
        print(f"  Delta = {r['delta_mPQ']:+.4f}")
        print()
    pos = (delta_df["delta_mPQ"] > 0).sum()
    tot = len(delta_df)
    print(f"FiLM helps in {pos}/{tot} backbones.")

Best overall run: 2026-02-14T105042_FilmRosieWeights-samhbaseline
  mPQ = 0.5865

Backbone SAM-H:
  Baseline mPQ = 0.5865
  Best FiLM mPQ = 0.5760
  Delta = -0.0105

FiLM helps in 0/1 backbones.
